<a href="https://colab.research.google.com/github/1heidi/inventory_2022/blob/inventory_update_2026/updating_inventory_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 GBC Inventory Update Pipeline

This notebook executes the automated text-mining and entity-extraction pipeline to update the GBC inventory. Because this repository relies on custom fine-tuned deep learning models built in 2022, new steps account for shifts in external packages, cloud environments, and Hugging Face Hub.

* **Hugging Face Hub Protocol Shifts:** The legacy `transformers` library engine used in the original codebase cannot handle modern Hugging Face URL caching and security redirects, causing network download failures. Cell 4 bypasses this entirely by establishing a strict local staging workflow, ensuring the weights are pulled down safely and verified via local checksums.
* **Modern Package & Environment Conflicts:** Modern runtime environments use updated Python stacks (Python 3.10+) and strict PyTorch serialization rules. These modern updates block or crash when trying to read legacy 2022 model checkpoints that contain custom team tracking objects (like `inventory_utils.custom_classes.Metrics`).
* **Safeguards:** To fix this without rewriting the core scripts, this notebook downgrades the base engine, injects repository paths directly into Python's runtime memory (Cell 5), and strictly maps checkpoint pointers directly to binary files. This allows use of the original 2022 fine-tuned assets within a cloud container.

### Execution Protocol

1. (Recommended) Runtime -> Change runtime type -> GPU.
2. Run **Cell 1** to initialize the underlying Python 3.8 environment.
3. **Manually click "Restart session"** at the top of Google Colab when Cell 1 finishes. Confirmation must be provided in Cell 1B.
4. Run **Cells 2 through 6 in absolute sequential order.** Do not skip steps or re-run cells out of order, and be sure to adjust the yml and login in config files for the new date range.


In [ ]:
# ==============================================================================
# CELL 1: ENGINE INITIALIZATION & PYTHON DOWNGRADE
# ==============================================================================
# Why it's being done differently:
# Modern Google Colab runtimes use newer Python versions (like Python 3.10+)
# that are incompatible with the legacy 2022 dependency tree. We force-install
# a native Python 3.8 Linux Conda environment to preserve execution stability.

#### 🛑 CRITICAL ACTION: You must click the "Restart session" banner at the top
#of Colab immediately after this cell finishes to force the notebook interface
#to switch over to the newly installed Python engine.

# 1. Download and install a native Python 3.8 Conda environment
!wget -qO installer.sh https://repo.anaconda.com/miniconda/Miniconda3-py38_4.12.0-Linux-x86_64.sh
!bash installer.sh -b -f -p /usr/local

# 2. Configure conda and enforce Python 3.8 + pip alignment
!conda config --set always_yes yes
!conda install -y -c conda-forge python=3.8 pip

# 3. Print verified version (Must confirm: Python 3.8.x)
!python --version

In [ ]:
# ==============================================================================
# CELL 1B: CONFIRM RESTART
# ==============================================================================

confirm = input(
    "Restart session completed? Type Y to continue: "
).strip()

if confirm != "Y":
    raise SystemExit(
        "STOPPED: You must type 'Y' after restarting the runtime."
    )

print("✔ Restart confirmed. Proceeding...")

In [ ]:
# ==============================================================================
# CELL 2: STORAGE MAPPING & DIRECTORY NAVIGATION
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/GitHub/inventory_2022

In [ ]:
# ==============================================================================
## CELL 2B: DOUBLE CHECK BRANCH
# ==============================================================================
%%bash
cd /content/drive/MyDrive/GitHub/inventory_2022

git branch --show-current

In [ ]:
# ==============================================================================
# CELL 3: LEGACY LIBRARY PINNING
# ==============================================================================
# Why it's being done differently:
# Modern versions of Hugging Face 'transformers' have deprecated the original
# 2022 internal tokenizing properties. Pinning these exact legacy versions
# protects the model from crashing during inference text tokenization.

!pip install tokenizers==0.11.4 transformers==4.16.2

In [ ]:
%%bash
# ==============================================================================
# CELL 4: PRODUCTION WEIGHTS REGISTRY & DEPENDENCY BOOTSTRAP
# ==============================================================================
# Why this cell exists:
#
# This pipeline depends on two distinct model systems:
#
# 1. Legacy PyTorch checkpoints (.pt files)
#    - Used directly by custom 2022 training/inference code
#    - Must be referenced via explicit file-path pointer files
#    - These pointer files are consumed by native Python open() calls
#
# 2. Hugging Face Transformer backbone models
#    - Required by AutoConfig / AutoModel loading in classification + NER steps
#    - Must exist as a fully materialized local directory OR valid HF repo access
#
# This cell ensures both systems are correctly initialized.
#
# Design constraints:
# - Avoid re-downloading existing large assets (efficiency + reproducibility)
# - Prevent pointer mismatches that break downstream Snakemake jobs
# - Ensure deterministic file paths for all model resolution steps
# - Guarantee compatibility between legacy PyTorch pipeline and HF Transformers
#
# Result:
# After this cell, all downstream pipeline steps (Snakemake rules) can safely:
# - Load .pt checkpoints via explicit path pointers
# - Load Transformer configs either locally or via resolved HF cache
# ==============================================================================
set -e  # fail fast if anything breaks

cd /content/drive/MyDrive/GitHub/inventory_2022 # Ensure we are in the correct directory

# ------------------------------------------------------------------------------
# 1. Repo setup (Directly execute commands from 'setup_for_updating' target)
# ------------------------------------------------------------------------------
python3.8 -m pip install -r requirements.txt
python3 -c "import nltk; nltk.download('punkt')"

# ------------------------------------------------------------------------------
# 2. CLASSIFIER WEIGHTS (PyTorch .pt)
# ------------------------------------------------------------------------------
mkdir -p out/classif_train_out/best

if [ ! -f out/classif_train_out/article_classifier.pt ]; then
    wget -O out/classif_train_out/article_classifier.pt \
    https://huggingface.co/globalbiodata/inventory/resolve/main/article_classifier.pt
fi

echo "5718a7f70becacb46d46501734c83aab81c86feec563594f6a25c116aa31b521 out/classif_train_out/article_classifier.pt" \
| sha256sum -c

echo "out/classif_train_out/article_classifier.pt" > out/classif_train_out/best/best_checkpt.txt

# ------------------------------------------------------------------------------
# 3. NER WEIGHTS (PyTorch .pt)
# ------------------------------------------------------------------------------
mkdir -p out/ner_train_out/best

if [ ! -f out/ner_train_out/named_entity_recognition.pt ]; then
    wget -O out/ner_train_out/named_entity_recognition.pt \
    https://huggingface.co/globalbiodata/inventory/resolve/main/named_entity_recognition.pt
fi

echo "dc0bc8b4929e33da52bc92e12720260b392421883889e0a36c809cb0b5c40f5d out/ner_train_out/named_entity_recognition.pt" \
| sha256sum -c

echo "out/ner_train_out/named_entity_recognition.pt" > out/ner_train_out/best/best_checkpt.txt

# ------------------------------------------------------------------------------
# 4. 🚨 CRITICAL FIX: HF TRANSFORMER BACKBONE (REQUIRED)
# ------------------------------------------------------------------------------
HF_MODEL_DIR="allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500"

mkdir -p "$HF_MODEL_DIR"

# Download ONLY if missing (safe + idempotent)
if [ ! -f "$HF_MODEL_DIR/config.json" ]; then
    echo "Downloading Hugging Face backbone model..."

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/config.json

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/pytorch_model.bin

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/tokenizer_config.json

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/vocab.json

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/merges.txt

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/special_tokens_map.json
fi

# ------------------------------------------------------------------------------
# 5. Sanity check
# ------------------------------------------------------------------------------
test -f "$HF_MODEL_DIR/config.json" || {
    echo "❌ HF backbone missing"
    exit 1
}

echo "✔ CELL 4 COMPLETE: all weights + HF backbone ready"


In [ ]:
# ==============================================================================
# CELL 5: RUNTIME ENVIRONMENT PATH INJECTION
# ==============================================================================
# Why it's being done differently:
# Your fine-tuned 2022 model weights (.pt files) contain specialized legacy
# custom tracking objects (like Metrics classes). When PyTorch unpickles these
# files during execution, it throws a ModuleNotFoundError because it can't find
# the source code modules. Injecting 'src' into Python's system path permanently
# bridges this map for all downstream execution steps.

import os
import sys
sys.path.append(os.path.abspath("src"))
print("Runtime paths synchronized successfully! Workspace ready for pipeline.")

In [ ]:
# ==============================================================================
# CELL 5B: LOCAL TRANSFORMER MODEL RESOLUTION (ROBUST VERSION)
# ==============================================================================
# Why this exists:
# Ensures the Hugging Face backbone model is fully available locally and
# prevents silent failures during AutoConfig / AutoModel loading.

import os
from transformers import AutoConfig

# ----------------------------------------------------------------------
# 1. Resolve model directory safely (Colab + Drive safe)
# ----------------------------------------------------------------------
# Correctly resolve MODEL_DIR by using the known absolute path from Cell 2/4 context
# The files are downloaded to /content/drive/MyDrive/GitHub/inventory_2022/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500
REPO_ROOT = "/content/drive/MyDrive/GitHub/inventory_2022"
HF_MODEL_RELATIVE_PATH = "allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500"
MODEL_DIR = os.path.join(REPO_ROOT, HF_MODEL_RELATIVE_PATH)


print("Resolved model path:", MODEL_DIR)

# ----------------------------------------------------------------------
# 2. Required HF files for a valid transformer model
# ----------------------------------------------------------------------
REQUIRED_FILES = [
    "config.json",
    "pytorch_model.bin",
    "vocab.json",
    "merges.txt",
    "tokenizer_config.json"
]

missing = [
    f for f in REQUIRED_FILES
    if not os.path.exists(os.path.join(MODEL_DIR, f))
]

if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f"Model directory missing: {MODEL_DIR}")

if missing:
    raise FileNotFoundError(
        "❌ Incomplete Hugging Face model folder.\n"
        f"Missing files: {missing}\n"
        "Fix CELL 4 download step."
    )

print("✔ Local HF model folder complete")

# ----------------------------------------------------------------------
# 3. Load config safely (no internet, no HF lookup)
# ----------------------------------------------------------------------
config = AutoConfig.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=False
)

print("✔ Transformer config loaded successfully")
print("Model type:", getattr(config, "model_type", "unknown"))

# Setting up Configurations

Before running the automated pipelines, first update the configuration file `config/update_inventory.yml`. It can be accessed in Google Drive, though you may need to download it and edit it in a text editor such as Notepad, then reupload it.

* **Europe PMC query publication date range**: These are stored as variables `query_from_date` and `query_to_date` in that file. Note that the dates are inclusive. For example to get papers published in 2022, both of those variables should be 2022.
* **Previous inventory file**: During strict deduplication and flagging for manual review, the results of the previous inventory are taken into account. Specify the location of the most recent inventory output file in the variable `previous_inventory`.

# Running the pipeline
---
Now, we are ready to run the pipeline. It will take several minutes or even over an hour. Job progression will be shown in the output.

In [ ]:
%%bash
# ==============================================================================
# CELL 6: LAUNCH INVENTORY PIPELINE - Minutes (GPU) or Hours (CPU)
# ==============================================================================

cd /content/drive/MyDrive/GitHub/inventory_2022
make update_inventory


# Selective Manual Review

After running the initial pipeline, the inventory has been flagged for selective manual review.

The file to be reviewed is located at:

`out/new_query/for_manual_review/predictions.csv`

Review the flagged columns according to the instruction sheet ([doi: 10.5281/zenodo.7768363](https://doi.org/10.5281/zenodo.7768363)), then place the manually reviewed file in the following folder:

`out/new_query/manually_reviewed/`

The file must still be named `predictions.csv`

# Processing Manual Review

Next, further processing is performed on the manually reviewed inventory.

In [ ]:
#  =============================================================================
#  CELL 7: RE-ANCHOR WORKING DIRECTORY
# ==============================================================================
%cd /content/drive/MyDrive/GitHub/inventory_2022

In [ ]:
# ==============================================================================
# CELL 8: PROGRAMMATIC PATCH FOR RATE-LIMIT PROTECTION DURING URL CHECKS
# ==============================================================================

!git checkout src/check_urls.py ## resets src/check_urls.py to git baseline in
# case want to modify the patch (e.g., changing 1.0 to 0.5 seconds)

import ast

file_path = "src/check_urls.py"

with open(file_path, "r") as f:
    content = f.read()

# 1. Prevent double-patching if already modified
if "time.sleep(1.0)" in content:
    print("ℹ️ check_urls.py is already patched. Skipping.")
else:
    target_str = "returned_dict = cast(dict, r.json())"

    if target_str in content:
        lines = content.splitlines(keepends=True)
        new_lines = []
        patched = False

        for line in lines:
            if target_str in line and not patched:
                indent = line[: len(line) - len(line.lstrip())]
                patch = (
                    f"{indent}import time\n"
                    f"{indent}time.sleep(1.0)\n"
                    f"{indent}try:\n"
                    f"{indent}    returned_dict = cast(dict, r.json())\n"
                    f"{indent}except Exception:\n"
                    f"{indent}    return None\n"
                )
                new_lines.append(patch)
                patched = True
            else:
                new_lines.append(line)

        new_content = "".join(new_lines)

        # 2. Test syntax in memory before saving
        try:
            ast.parse(new_content)
            with open(file_path, "w") as f:
                f.write(new_content)
            print("✅ Successfully patched check_urls.py with valid syntax!")
        except SyntaxError as e:
            print(f"❌ Patch cancelled! Syntax error detected on line {e.lineno}: {e.msg}")
            print("Please check the lines immediately surrounding target_str in check_urls.py.")
    else:
        print("⚠️ Target line not found in file.")

In [ ]:
# ==============================================================================
# CELL 9: AUGMENTATION AND FINAL INVENTORY FILE CREATION
# ==============================================================================

# 1. Patch the Snakemake rule in snakemake/shared_rules.smk to include the required --file flag
!sed -i 's/epmc_meta \+{input}/epmc_meta --file {input}/g' snakemake/shared_rules.smk || true

# 2. Execute the final inventory update rules (get_epmc_meta -> process_countries)
!make process_manually_reviewed_update

## Final inventory

The final inventory, including names, URLS, and metadata is found in the file:
*    `out/new_query/processed_countries/predictions.csv`